# 🇮🇳 Hindi Corpus Collection (Model H)

**Sources**: Sangraha Verified (automated) + Wikipedia Dump (manual)

**Target**: 500M tokens total (≥20% manual = ≥100M manual tokens)

**Sangraha Verified has 12.6B Hindi tokens** — we only need ~400M automated.

**⚡ Direct File-by-File Downloader: Zero RAM overhead, 100% reliable, auto-stops at target.**

## 0. Setup + Resume

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_BASE = '/content/drive/MyDrive/24KaratAu/hindi'
for d in ['raw/wiki_dump', 'cleaned/manual', 'cleaned/automated_nonweb',
          'cleaned/automated_web', 'splits', 'tokenizer']:
    os.makedirs(f'{DRIVE_BASE}/{d}', exist_ok=True)
print('✅ Drive directories created')

In [ ]:
!pip install -q pandas pyarrow sentencepiece
print('✅ Dependencies installed')

In [ ]:
import json, hashlib, re, random, bz2, os, gc, sys
import urllib.request
import pandas as pd
import xml.etree.ElementTree as ET
from pathlib import Path
from collections import Counter

LANGUAGE = 'hindi'
MIN_CHARS = 100
MIN_DEV_RATIO = 0.5
SHARD_SIZE = 10000

CHARS_PER_TOKEN = 6
AUTO_TOKEN_TARGET  = 400_000_000
AUTO_CHAR_TARGET   = AUTO_TOKEN_TARGET * CHARS_PER_TOKEN

NONWEB_DIR = f'{DRIVE_BASE}/cleaned/automated_nonweb'
MANUAL_DIR = f'{DRIVE_BASE}/cleaned/manual'
WEB_DIR    = f'{DRIVE_BASE}/cleaned/automated_web'
SPLITS_DIR = f'{DRIVE_BASE}/splits'

def devanagari_ratio(text):
    if not text: return 0.0
    return sum(1 for c in text if '\u0900' <= c <= '\u097F') / len(text)

def clean_web_text(text):
    text = re.sub(r'https?://\S+', '', text)
    text = re.sub(r'\S+@\S+\.\S+', '', text)
    text = re.sub(r'&[a-zA-Z]+;', ' ', text)
    text = re.sub(r'&#\d+;', ' ', text)
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()

def text_hash_int(text):
    return int(hashlib.md5(text.encode('utf-8')).hexdigest()[:16], 16)

# ── RESUME ──
all_hashes = set()
existing_auto_chars = 0
existing_manual_chars = 0
existing_docs = 0
for d in [NONWEB_DIR, MANUAL_DIR, WEB_DIR]:
    for sf in sorted(Path(d).glob('*.jsonl')):
        with open(sf, 'r', encoding='utf-8') as f:
            for line in f:
                if not line.strip(): continue
                try:
                    doc = json.loads(line)
                    h = doc.get('text_sha256', '')
                    if h:
                        all_hashes.add(int(h[:16], 16) if len(h) > 16 else int(h, 16))
                    existing_docs += 1
                    c = doc.get('clean_text_characters', len(doc.get('text','')))
                    if doc.get('collection_type') == 'manual':
                        existing_manual_chars += c
                    else:
                        existing_auto_chars += c
                except: pass

if existing_docs > 0:
    print(f'🔄 RESUMING: {existing_docs:,} docs | auto: ~{existing_auto_chars//CHARS_PER_TOKEN:,} tok | manual: ~{existing_manual_chars//CHARS_PER_TOKEN:,} tok')
else:
    print('📦 Fresh start')
print(f'✅ Ready | Target: {AUTO_TOKEN_TARGET//1_000_000}M auto tokens')

---
## Step 1: Sangraha PDF + ASR (automated, zero wiki overlap)

Direct File-by-File Pipeline: Downloads 1 parquet file at a time, processes with pandas, frees memory.

In [ ]:
auto_chars_so_far = existing_auto_chars
nonweb_docs = []
nonweb_shard_idx = len(list(Path(NONWEB_DIR).glob('*.jsonl')))
nonweb_total_chars = 0
nonweb_total_docs = 0
type_counts = Counter()

def flush_nonweb():
    global nonweb_docs, nonweb_shard_idx
    if not nonweb_docs: return
    sf = f'{NONWEB_DIR}/nonweb_shard_{nonweb_shard_idx:04d}.jsonl'
    with open(sf, 'w', encoding='utf-8') as f:
        for d in nonweb_docs:
            f.write(json.dumps(d, ensure_ascii=False) + '\n')
    print(f'  ✅ Shard saved to Drive: {Path(sf).name} ({len(nonweb_docs):,} docs)')
    nonweb_docs = []
    nonweb_shard_idx += 1
    gc.collect()

print(f'Fetching Sangraha verified Hindi Parquet file listing...')
api_url = 'https://huggingface.co/api/datasets/ai4bharat/sangraha'
req = urllib.request.Request(api_url, headers={'User-Agent': 'Mozilla/5.0'})
with urllib.request.urlopen(req) as resp:
    repo_data = json.loads(resp.read().decode())
    all_files = [f.get('rfilename') for f in repo_data.get('siblings', [])]

hin_files = sorted([f for f in all_files if f.startswith('verified/hin/') and f.endswith('.parquet')])
print(f'✅ Found {len(hin_files)} Parquet files for verified Hindi.')
print(f'Target: {AUTO_CHAR_TARGET:,} auto chars (~{AUTO_TOKEN_TARGET//1_000_000}M tokens)\n')

if auto_chars_so_far >= AUTO_CHAR_TARGET:
    print('✅ Already at target! Skipping Step 1.')
else:
    for idx, rel_path in enumerate(hin_files):
        if auto_chars_so_far >= AUTO_CHAR_TARGET:
            print('🎯 HIT AUTO TARGET! Stopping download.')
            break
            
        file_name = Path(rel_path).name
        temp_local = f'/content/{file_name}'
        dl_url = f'https://huggingface.co/datasets/ai4bharat/sangraha/resolve/main/{rel_path}'
        
        print(f'📥 [{idx+1}/{len(hin_files)}] Downloading {file_name}...')
        urllib.request.urlretrieve(dl_url, temp_local)
        
        # Load single file into pandas (instant, ~150MB RAM max)
        df = pd.read_parquet(temp_local)
        print(f'   Processing {len(df):,} records...')
        
        for row in df.itertuples():
            doc_type = str(getattr(row, 'type', '')).strip().lower()
            type_counts[doc_type] += 1
            if 'web' in doc_type: continue  # PDF/ASR only in Step 1
            
            text = str(getattr(row, 'text', ''))
            if not text or text == 'nan': continue
            cleaned = clean_web_text(text)
            if len(cleaned) < MIN_CHARS or devanagari_ratio(cleaned) < MIN_DEV_RATIO: continue
            
            hi = text_hash_int(cleaned)
            if hi in all_hashes: continue
            all_hashes.add(hi)
            
            nonweb_docs.append({
                'doc_id': f'{hi:x}',
                'source': 'sangraha', 'source_type': f'hf_sangraha_{doc_type}',
                'collection_type': 'automated', 'language': LANGUAGE,
                'text': cleaned, 'clean_text_characters': len(cleaned),
                'devanagari_ratio': round(devanagari_ratio(cleaned), 4),
                'text_sha256': f'{hi:x}', 'sangraha_type': doc_type,
            })
            nonweb_total_chars += len(cleaned)
            nonweb_total_docs += 1
            auto_chars_so_far += len(cleaned)
            
            if len(nonweb_docs) >= SHARD_SIZE: flush_nonweb()
            if auto_chars_so_far >= AUTO_CHAR_TARGET:
                break
                
        # Clean up temp file & free memory immediately
        if os.path.exists(temp_local):
            os.remove(temp_local)
        del df
        gc.collect()
        
        pct = auto_chars_so_far / AUTO_CHAR_TARGET * 100
        print(f'   ✅ Processed {file_name} | Total docs: {nonweb_total_docs:,} | Auto chars: {auto_chars_so_far:,} ({pct:.1f}% of target)\n')
        
    flush_nonweb()

print(f'\nStep 1 Complete: {nonweb_total_docs:,} docs, {nonweb_total_chars:,} chars')
print(f'Auto total so far: ~{auto_chars_so_far//CHARS_PER_TOKEN:,} tokens')

---
## Step 2: Wikipedia Dump (manual)

In [ ]:
WIKI_DUMP = '/content/hiwiki-latest-pages-articles.xml.bz2'
if not os.path.exists(WIKI_DUMP):
    print('Downloading Hindi Wikipedia dump (~600MB)...')
    !wget -q --show-progress -O {WIKI_DUMP} \
        'https://dumps.wikimedia.org/hiwiki/latest/hiwiki-latest-pages-articles.xml.bz2'
print(f'✅ Dump: {os.path.getsize(WIKI_DUMP)/(1024*1024):.1f} MB')
!cp {WIKI_DUMP} {DRIVE_BASE}/raw/wiki_dump/ 2>/dev/null || true

In [ ]:
WIKI_MARKUP = [
    (re.compile(r'\{\{[^}]*\}\}'), ''),
    (re.compile(r'\[\[Category:[^\]]*\]\]'), ''),
    (re.compile(r'\[\[श्रेणी:[^\]]*\]\]'), ''),
    (re.compile(r'\[\[वर्ग:[^\]]*\]\]'), ''),
    (re.compile(r'\[\[[^|\]]*\|([^\]]*)\]\]'), r'\1'),
    (re.compile(r'\[\[([^\]]*)\]\]'), r'\1'),
    (re.compile(r'\[https?://[^\s\]]*\s*([^\]]*)\]'), r'\1'),
    (re.compile(r'\[https?://[^\]]*\]'), ''),
    (re.compile(r"'{2,5}"), ''),
    (re.compile(r'<ref[^>]*>.*?</ref>', re.DOTALL), ''),
    (re.compile(r'<ref[^/>]*/>'), ''),
    (re.compile(r'<[^>]+>'), ''),
    (re.compile(r'&[a-zA-Z]+;'), ' '),
    (re.compile(r'\[\d+\]'), ''),
]
TABLE_PAT = [
    (re.compile(r'^\s*\{\|.*$', re.MULTILINE), ''),
    (re.compile(r'^\s*\|\}.*$', re.MULTILINE), ''),
    (re.compile(r'^\s*\|-.*$', re.MULTILINE), ''),
    (re.compile(r'^\s*!.*$', re.MULTILINE), ''),
    (re.compile(r'^\|.*$', re.MULTILINE), ''),
]
SKIP_SEC = {'सन्दर्भ','संदर्भ','बाहरी कड़ियाँ','बाहरी कड़ी','श्रेणी',
            'इन्हें भी देखें','यह भी देखें','स्रोत','टिप्पणी','नोट',
            'references','external links','see also','notes'}

def clean_wiki(text):
    for p,r in TABLE_PAT: text = p.sub(r, text)
    for p,r in WIKI_MARKUP: text = p.sub(r, text)
    lines, result, skip = text.split('\n'), [], False
    for line in lines:
        s = line.strip().lower()
        if s in SKIP_SEC: skip = True; continue
        if re.match(r'^={2,}.*={2,}$', s): skip = False
        if not skip: result.append(line)
    text = '\n'.join(result)
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()

with bz2.open(WIKI_DUMP, 'rt', encoding='utf-8') as f:
    header = f.read(5000)
    ns_match = re.search(r'xmlns="(http://www\.mediawiki\.org/xml/export-[^"]+)"', header)
    NS = '{' + ns_match.group(1) + '}' if ns_match else ''
print(f'✅ XML namespace: {NS or "none"}')

In [ ]:
man_docs, man_chars, man_count = [], 0, 0
man_shard = len(list(Path(MANUAL_DIR).glob('*.jsonl')))
seen, skipped_dup, rejected = 0, 0, 0

print('Processing Hindi Wikipedia dump (ALL articles)...')

with bz2.open(WIKI_DUMP, 'rt', encoding='utf-8') as f:
    for event, elem in ET.iterparse(f, events=('end',)):
        tag = elem.tag.split('}')[-1] if '}' in elem.tag else elem.tag
        if tag != 'page': continue
        ns_el = elem.find(f'{NS}ns') if NS else elem.find('ns')
        if ns_el is None or ns_el.text != '0': elem.clear(); continue
        redirect = elem.find(f'{NS}redirect') if NS else elem.find('redirect')
        if redirect is not None: elem.clear(); continue
        title_el = elem.find(f'{NS}title') if NS else elem.find('title')
        id_el = elem.find(f'{NS}id') if NS else elem.find('id')
        title = title_el.text if title_el is not None else ''
        pid = id_el.text if id_el is not None else ''
        rev = elem.find(f'{NS}revision') if NS else elem.find('revision')
        raw = ''
        if rev is not None:
            te = rev.find(f'{NS}text') if NS else rev.find('text')
            if te is not None and te.text: raw = te.text
        elem.clear()
        if not raw.strip(): continue
        seen += 1
        cleaned = clean_wiki(raw)
        if len(cleaned) < MIN_CHARS or devanagari_ratio(cleaned) < MIN_DEV_RATIO:
            rejected += 1; continue
        hi = text_hash_int(cleaned)
        if hi in all_hashes: skipped_dup += 1; continue
        all_hashes.add(hi)
        man_docs.append({
            'doc_id': f'{pid}_{title}',
            'title': title, 'wiki_page_id': pid,
            'source': 'Hindi Wikipedia Dump', 'source_type': 'wikipedia_dump',
            'collection_type': 'manual', 'language': LANGUAGE,
            'text': cleaned, 'clean_text_characters': len(cleaned),
            'devanagari_ratio': round(devanagari_ratio(cleaned), 4),
            'text_sha256': f'{hi:x}',
        })
        man_chars += len(cleaned); man_count += 1
        if len(man_docs) >= SHARD_SIZE:
            sf = f'{MANUAL_DIR}/wiki_shard_{man_shard:04d}.jsonl'
            with open(sf,'w',encoding='utf-8') as out:
                for d in man_docs: out.write(json.dumps(d,ensure_ascii=False)+'\n')
            print(f'  ✅ {Path(sf).name}: {len(man_docs):,} docs')
            man_docs = []; man_shard += 1; gc.collect()
        if seen % 10000 == 0:
            print(f'  Seen:{seen:,} Accept:{man_count:,} Chars:{man_chars:,}')

if man_docs:
    sf = f'{MANUAL_DIR}/wiki_shard_{man_shard:04d}.jsonl'
    with open(sf,'w',encoding='utf-8') as out:
        for d in man_docs: out.write(json.dumps(d,ensure_ascii=False)+'\n')
    print(f'  ✅ {Path(sf).name}: {len(man_docs):,} docs')
    man_docs = []; gc.collect()

print(f'\n{"="*60}')
print(f'WIKI COMPLETE: {man_count:,} articles, {man_chars:,} chars (~{man_chars//CHARS_PER_TOKEN:,} tokens)')
print(f'{"="*60}')

---
## Step 3: Sangraha Web (fill remaining automated if needed)

In [ ]:
web_docs, web_chars, web_count = [], 0, 0
web_shard_idx = len(list(Path(WEB_DIR).glob('*.jsonl')))

def flush_web():
    global web_docs, web_shard_idx
    if not web_docs: return
    sf = f'{WEB_DIR}/web_shard_{web_shard_idx:04d}.jsonl'
    with open(sf,'w',encoding='utf-8') as f:
        for d in web_docs: f.write(json.dumps(d,ensure_ascii=False)+'\n')
    print(f'  ✅ {Path(sf).name}: {len(web_docs):,} docs')
    web_docs = []; web_shard_idx += 1; gc.collect()

if auto_chars_so_far >= AUTO_CHAR_TARGET:
    print(f'✅ Already have {auto_chars_so_far//CHARS_PER_TOKEN:,} auto tokens — skipping web!')
else:
    remaining = AUTO_CHAR_TARGET - auto_chars_so_far
    print(f'Need {remaining:,} more auto chars (~{remaining//CHARS_PER_TOKEN:,} tokens)...')
    print('Direct File-by-File Pipeline: Sangraha Web verified/hin...')
    for idx, rel_path in enumerate(hin_files):
        if auto_chars_so_far >= AUTO_CHAR_TARGET: break
        file_name = Path(rel_path).name
        temp_local = f'/content/{file_name}'
        dl_url = f'https://huggingface.co/datasets/ai4bharat/sangraha/resolve/main/{rel_path}'
        urllib.request.urlretrieve(dl_url, temp_local)
        df = pd.read_parquet(temp_local)
        for row in df.itertuples():
            doc_type = str(getattr(row, 'type', '')).strip().lower()
            if 'web' not in doc_type: continue
            text = str(getattr(row, 'text', ''))
            if not text or text == 'nan': continue
            cleaned = clean_web_text(text)
            if len(cleaned) < MIN_CHARS or devanagari_ratio(cleaned) < MIN_DEV_RATIO: continue
            hi = text_hash_int(cleaned)
            if hi in all_hashes: continue
            all_hashes.add(hi)
            web_docs.append({
                'doc_id': f'{hi:x}',
                'source': 'sangraha', 'source_type': 'hf_sangraha_web',
                'collection_type': 'automated', 'language': LANGUAGE,
                'text': cleaned, 'clean_text_characters': len(cleaned),
                'devanagari_ratio': round(devanagari_ratio(cleaned), 4),
                'text_sha256': f'{hi:x}',
            })
            web_chars += len(cleaned); web_count += 1
            auto_chars_so_far += len(cleaned)
            if len(web_docs) >= SHARD_SIZE: flush_web()
            if auto_chars_so_far >= AUTO_CHAR_TARGET: break
        if os.path.exists(temp_local): os.remove(temp_local)
        del df; gc.collect()
    flush_web()
    print(f'Web: {web_count:,} docs, {web_chars:,} chars')

In [ ]:
manual_char_total, auto_char_total = 0, 0
manual_total, auto_total = 0, 0
for d in [NONWEB_DIR, MANUAL_DIR, WEB_DIR]:
    for sf in Path(d).glob('*.jsonl'):
        with open(sf,'r',encoding='utf-8') as f:
            for line in f:
                if not line.strip(): continue
                doc = json.loads(line)
                c = doc.get('clean_text_characters', len(doc.get('text','')))
                if doc.get('collection_type') == 'manual':
                    manual_total += 1; manual_char_total += c
                else:
                    auto_total += 1; auto_char_total += c
tot = manual_char_total + auto_char_total
print(f'\n{"="*60}')
print(f'COLLECTION COMPLETE')
print(f'  Manual:    {manual_total:,} docs, ~{manual_char_total//CHARS_PER_TOKEN:,} tokens')
print(f'  Automated: {auto_total:,} docs, ~{auto_char_total//CHARS_PER_TOKEN:,} tokens')
print(f'  Total:     ~{tot//CHARS_PER_TOKEN:,} tokens')
print(f'  Manual %:  {manual_char_total/max(1,tot)*100:.1f}%')
print(f'{"="*60}')

---
## Step 4: Split + Tokenizer

In [ ]:
random.seed(42)
out = {s: open(f'{SPLITS_DIR}/{s}.txt','w',encoding='utf-8') for s in ['train','val','test']}
jout = {s: open(f'{SPLITS_DIR}/{s}.jsonl','w',encoding='utf-8') for s in ['train','val','test']}
counts = {'train':0,'val':0,'test':0}

for d in [MANUAL_DIR, NONWEB_DIR, WEB_DIR]:
    for sf in sorted(Path(d).glob('*.jsonl')):
        with open(sf,'r',encoding='utf-8') as f:
            for line in f:
                if not line.strip(): continue
                r = random.random()
                s = 'train' if r < 0.90 else ('val' if r < 0.95 else 'test')
                jout[s].write(line)
                doc = json.loads(line)
                t = doc.get('text','')
                if t.strip(): out[s].write(t.strip()+'\n\n')
                counts[s] += 1

for f in out.values(): f.close()
for f in jout.values(): f.close()
for s in ['train','val','test']: print(f'  {s}: {counts[s]:,} docs')
print('✅ Splits saved')

In [ ]:
import sentencepiece as spm
MPREFIX = f'{DRIVE_BASE}/tokenizer/hindi_bpe_32k'
TRAIN_TXT = f'{SPLITS_DIR}/train.txt'
mb = os.path.getsize(TRAIN_TXT)/(1024*1024)
print(f'Training BPE on {mb:.1f} MB...')
if mb < 0.01:
    print('⚠️ Empty!')
else:
    spm.SentencePieceTrainer.train(
        input=TRAIN_TXT, model_prefix=MPREFIX, vocab_size=32000,
        model_type='bpe', character_coverage=0.9995, num_threads=4,
        max_sentence_length=16384, input_sentence_size=1_000_000,
        shuffle_input_sentence=True,
        pad_id=0, unk_id=1, bos_id=2, eos_id=3,
        pad_piece='<pad>', unk_piece='<unk>', bos_piece='<bos>', eos_piece='<eos>',
        split_by_unicode_script=True, split_by_whitespace=True,
        split_digits=True, byte_fallback=True,
    )
    sp = spm.SentencePieceProcessor(); sp.load(f'{MPREFIX}.model')
    print(f'✅ Vocab: {sp.get_piece_size():,}')
    print(sp.encode('भारत एक दक्षिण एशियाई देश है।', out_type=str))

In [ ]:
if os.path.exists(f'{MPREFIX}.model'):
    sp = spm.SentencePieceProcessor(); sp.load(f'{MPREFIX}.model')
    token_counts = {}
    for sn in ['train','val','test']:
        tt, td = 0, 0
        with open(f'{SPLITS_DIR}/{sn}.txt','r',encoding='utf-8') as f:
            buf = []
            for line in f:
                if line.strip(): buf.append(line.strip())
                elif buf:
                    tt += len(sp.encode(' '.join(buf))); td += 1; buf = []
            if buf: tt += len(sp.encode(' '.join(buf))); td += 1
        token_counts[sn] = {'docs': td, 'tokens': tt}
        print(f'  {sn}: {td:,} docs → {tt:,} tokens')
    TOTAL = sum(v['tokens'] for v in token_counts.values())
    summary = {
        'language': LANGUAGE,
        'manual_docs': manual_total, 'manual_chars': manual_char_total,
        'automated_docs': auto_total, 'automated_chars': auto_char_total,
        'total_docs': manual_total+auto_total, 'total_chars': tot,
        'manual_percentage': round(manual_char_total/max(1,tot)*100, 2),
        'token_counts': token_counts, 'total_tokens': TOTAL,
        'vocab_size': sp.get_piece_size(),
    }
    with open(f'{DRIVE_BASE}/corpus_summary.json','w') as f:
        json.dump(summary, f, indent=2)
    print(f'\n🎉 HINDI DONE! {TOTAL:,} tokens | Manual: {manual_char_total/max(1,tot)*100:.1f}%')
else:
    print('⚠️ No tokenizer.')